In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from tqdm import tqdm
import json
import textwrap
import sys
import os

In [2]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session (or once on a local machine).
# It mounts Drive (Colab only) and adds the repo root to sys.path so all
# project imports work correctly.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Add the repo root to the module search path
sys.path.insert(0, os.path.abspath(".."))   # works locally from notebooks/
                                             # adjust to REPO_ROOT on Colab

import config as cfg
cfg.make_dirs()
cfg.print_config()

Mounted at /content/drive


In [3]:
from model import GPT, GPTConfig, GPTWithKVCache
config = GPTConfig()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:", device)

In [7]:
RUN = 0

In [8]:
if RUN ==0 :
  from datasets import load_dataset

  # dataset = load_dataset("databricks/databricks-dolly-15k")
  dataset = load_dataset(cfg.SFT_DATASET_NAME)
  sft_data = dataset["train"][0:cfg.SFT_DATA_SIZE]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [10]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    cfg.TOKENIZER_VOCAB,
    cfg.TOKENIZER_MERGES
)

eos_id = tokenizer.token_to_id("</s>")

In [11]:
from dpo.dataset_generation import (
    extract_prompt,
    generate_candidates,
    score_response,
    generate_dpo_dataset,
    save_dpo_dataset,
)
from generation.sampler import generate, encode_prompt

cuda


In [13]:
if device == "cpu":
  torch.set_num_threads(2)

In [14]:

if device == "cpu":
  ckpt = torch.load(cfg.SFT_FINAL_CKPT, map_location="cpu")
else:
  ckpt = torch.load(cfg.SFT_FINAL_CKPT)

model.load_state_dict(ckpt["model"])

# model.eval()
print("Model loaded successfully")

Model loaded successfully


In [22]:
DPO_PATH = cfg.DPO_DATASET

In [28]:
test = generate_dpo_dataset(model, tokenizer, sft_data["text"][0:5])

  0%|          | 0/5 [00:00<?, ?it/s]

Write a short story for kids.

One day, a little girl named Lily found a needle in her room.

Response:



 20%|██        | 1/5 [00:03<00:13,  3.34s/it]

Write a short story for kids.

Once upon a time, there was a little car named Beep.

Response:



 40%|████      | 2/5 [00:06<00:09,  3.30s/it]

...

 60%|██████    | 3/5 [00:10<00:06,  3.48s/it]

Write a short story for kids.

Once upon a time, in a land full of trees, there was a little cherry tree.

Response:



 80%|████████  | 4/5 [00:14<00:03,  3.64s/it]

Write a short story for kids.

Once upon a time, there was a little girl named Lily.

Response:



100%|██████████| 5/5 [00:17<00:00,  3.48s/it]


In [29]:
test[0]

{'prompt': 'Write a short story for kids.\n\nOne day, a little girl named Lily found a needle in her room.\n\nResponse:\n',
 'chosen': 'One day, a little fish named Fin was swimming near the shore. He saw a big crab and wanted to be friends. "Hi, I am Fin. Do you want to play?" asked the little fish. The crab looked at Fin and said, "No, I don\'t want to play. I am cold and I don\'t feel fine."\n\nFin felt sad but wanted to help the crab feel better. He swam away and thought of a plan. He remembered that the sun could make things warm. So, Fin swam to the top of the water and called to the sun, "Please, sun, help my new friend feel fine and not freeze!"\n\nThe sun heard Fin\'s call and shone its warm light on the shore. The crab started to feel better and not so cold. He saw Fin and said, "Thank you, little fish, for making me feel fine. I don\'t feel like I will freeze now. Let\'s play together!" And so, Fin and the crab played and became good friends.',
 'rejected': 'Once upon a time